In [8]:
import pandas as pd
import numpy as np

# Load your GPS file
df_original = pd.read_csv(r"C:\Arjun\Thesis\data\20200421_170039-sunset1\sunset1_gps_v2.csv")
df_original = df_original.sort_values('elapsed_time_ts').reset_index(drop=True)

In [9]:
df.head(5)

,latitude,longitude,elapsed_time,elapsed_time_ts
0,-27.506620,152.911022,0,1.587453e+09
1,-27.506660,152.911140,7,1.587453e+09
2,-27.506688,152.911283,9,1.587453e+09
3,-27.506725,152.911476,11,1.587453e+09
4,-27.506745,152.911589,12,1.587453e+09


In [10]:
def interpolate_gps_by_timestamp(new_timestamps_ts, df_original):
    """
    Interpolate GPS coordinates for new timestamps (in elapsed_time_ts format)
    
    Parameters:
    new_timestamps_ts: list of Unix timestamps (same as elapsed_time_ts in original)
    df_original: DataFrame with original data
    
    Returns:
    DataFrame with new_timestamp_ts, latitude, longitude
    """
    times = df_original['elapsed_time_ts'].values
    lats = df_original['latitude'].values
    lons = df_original['longitude'].values
    
    results = []
    
    for target_ts in new_timestamps_ts:
        # Find where this timestamp would fit
        idx = np.searchsorted(times, target_ts)
        
        if idx == 0:
            # Before first point
            lat, lon = lats[0], lons[0]
            method = "first_point"
        elif idx >= len(times):
            # After last point
            lat, lon = lats[-1], lons[-1]
            method = "last_point"
        else:
            # Check for exact match
            if abs(times[idx] - target_ts) < 1e-6:
                lat, lon = lats[idx], lons[idx]
                method = "exact_match"
            else:
                # Interpolate between idx-1 and idx
                t0, t1 = times[idx-1], times[idx]
                lat0, lat1 = lats[idx-1], lats[idx]
                lon0, lon1 = lons[idx-1], lons[idx]
                
                # Linear interpolation
                ratio = (target_ts - t0) / (t1 - t0)
                lat = lat0 + ratio * (lat1 - lat0)
                lon = lon0 + ratio * (lon1 - lon0)
                method = f"interpolated (between {t0} and {t1})"
        
        results.append({
            'new_timestamp_ts': target_ts,
            'latitude': round(lat, 8),
            'longitude': round(lon, 8),
            'method': method
        })

    return pd.DataFrame(results)


In [12]:
new_timestamps = [
1587452718,
1587453275,
1587452846,
1587453117,
1587452831,
1587453151]
print(f"Using {len(new_timestamps)} manually entered timestamps")

# Perform interpolation
result_df = interpolate_gps_by_timestamp(new_timestamps, df_original)
output_file = 'interpolated_coordinates.csv'
#result_df[['new_timestamp_ts', 'latitude', 'longitude']].to_csv(output_file, index=False)
result_df.to_csv(output_file, index=False)
print(f"\nResults saved to '{output_file}'")

Using 6 manually entered timestamps

Results saved to 'interpolated_coordinates.csv'
